In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

In [2]:
DATASET_DIR = r"F:\work\python\image frequency\landsat\categorized_dataset1"

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(256,256),
    batch_size=16
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(256,256),
    batch_size=16
)

class_names = train_ds.class_names
print("Classes:", class_names)

Found 1369 files belonging to 3 classes.
Using 1096 files for training.
Found 1369 files belonging to 3 classes.
Using 273 files for validation.
Classes: ['high', 'low', 'medium']


In [3]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [4]:
def build_cnn():

    model = models.Sequential([
        layers.Input(shape=(256,256,3)),

        layers.Conv2D(32,3,activation='relu'),
        layers.MaxPooling2D(),

        layers.Conv2D(64,3,activation='relu'),
        layers.MaxPooling2D(),

        layers.Conv2D(128,3,activation='relu'),
        layers.GlobalAveragePooling2D(),

        layers.Dense(128, activation='relu'),
        layers.Dense(3, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [5]:
def build_resnet():

    base = tf.keras.applications.ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(256,256,3)
    )

    for layer in base.layers[:-20]:
        layer.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(128, activation='relu')(x)
    output = layers.Dense(3, activation='softmax')(x)

    model = tf.keras.Model(inputs=base.input, outputs=output)

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [6]:
def build_efficientnet():

    base = tf.keras.applications.EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(256,256,3)
    )

    for layer in base.layers[:-20]:
        layer.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(128, activation='relu')(x)
    output = layers.Dense(3, activation='softmax')(x)

    model = tf.keras.Model(inputs=base.input, outputs=output)

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [7]:
def build_densenet():

    base = tf.keras.applications.DenseNet121(
        weights='imagenet',
        include_top=False,
        input_shape=(256,256,3)
    )

    # Freeze most layers
    for layer in base.layers[:-20]:
        layer.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(128, activation='relu')(x)
    output = layers.Dense(3, activation='softmax')(x)

    model = tf.keras.Model(inputs=base.input, outputs=output)

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [12]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=6,
        restore_best_weights=True
    )
]

In [13]:
EPOCHS = 10

cnn = build_cnn()
cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

resnet = build_resnet()
resnet.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

effnet = build_efficientnet()
effnet.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

densenet = build_densenet()
densenet.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

Epoch 1/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 19s 263ms/step - accuracy: 0.5785 - loss: 2.0651 - val_accuracy: 0.7179 - val_loss: 0.6514
Epoch 2/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 17s 245ms/step - accuracy: 0.6925 - loss: 0.6428 - val_accuracy: 0.6777 - val_loss: 0.6915
Epoch 3/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 17s 241ms/step - accuracy: 0.7308 - loss: 0.6184 - val_accuracy: 0.7436 - val_loss: 0.5860
Epoch 4/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 17s 243ms/step - accuracy: 0.7181 - loss: 0.6027 - val_accuracy: 0.7473 - val_loss: 0.5785
Epoch 5/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 17s 247ms/step - accuracy: 0.7308 - loss: 0.6071 - val_accuracy: 0.7729 - val_loss: 0.5697
Epoch 6/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 17s 242ms/step - accuracy: 0.7582 - loss: 0.5732 - val_accuracy: 0.7729 - val_loss: 0.5227
Epoch 7/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 17s 243ms/step - accuracy: 0.7290 - loss: 0.5782 - val_accuracy: 0.8059 - val_loss: 0.5365
Epoch 8/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 17s 243ms/step - accuracy: 0.7445 - loss: 0.5475 - val_accu

In [14]:
def evaluate_per_class(model, dataset, class_names):

    y_true = []
    y_pred = []

    for images, labels in dataset:
        preds = model.predict(images)
        preds = np.argmax(preds, axis=1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    results = {}

    for i, cls in enumerate(class_names):
        idx = np.where(y_true == i)[0]

        if len(idx) == 0:
            acc = 0
        else:
            acc = np.mean(y_pred[idx] == y_true[idx])

        results[cls.upper()] = acc

    return results

In [16]:
import numpy as np

cnn_res = evaluate_per_class(cnn, val_ds, class_names)
resnet_res = evaluate_per_class(resnet, val_ds, class_names)
effnet_res = evaluate_per_class(effnet, val_ds, class_names)
densenet_res = evaluate_per_class(densenet, val_ds, class_names)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 562ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 558ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 549ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 548ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 539ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 571ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 552ms/step
1/1 ━

In [17]:
for cls in class_names:

    cls_upper = cls.upper()

    print(f"\n{cls_upper}:")

    print(f"CNN: {cnn_res.get(cls_upper, 0)}")
    print(f"ResNet: {resnet_res.get(cls_upper, 0)}")
    print(f"EfficientNet: {effnet_res.get(cls_upper, 0)}")
    print(f"DenseNet: {densenet_res.get(cls_upper, 0)}")


HIGH:
CNN: 0.810126582278481
ResNet: 0.8860759493670886
EfficientNet: 0.8354430379746836
DenseNet: 0.8987341772151899

LOW:
CNN: 0.5098039215686274
ResNet: 0.8235294117647058
EfficientNet: 0.39215686274509803
DenseNet: 0.8823529411764706

MEDIUM:
CNN: 0.8601398601398601
ResNet: 0.8811188811188811
EfficientNet: 0.951048951048951
DenseNet: 0.8741258741258742
